In [3]:
import numpy as np
import json
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

# load scaler params 
with open('./models/scaler_params.json', 'r') as f:
    scaler_params = json.load(f)

FEATURE_COLS = scaler_params['feature_cols']
classes      = scaler_params['classes']
mean         = np.array(scaler_params['mean'])
scale        = np.array(scaler_params['scale'])

# rebuild encoder from saved classes
encoder = LabelEncoder()
encoder.classes_ = np.array(classes)

# load tflite model 
interpreter = tf.lite.Interpreter(model_path='models/asl_model.tflite')
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# predict function
def predict_sign(reading: dict):
    sample = np.array([[reading[col] for col in FEATURE_COLS]])
    sample_scaled = (sample - mean) / scale  # manual scaling, no scaler object needed
    sample_scaled = sample_scaled.astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], sample_scaled)
    interpreter.invoke()
    probs     = interpreter.get_tensor(output_details[0]['index'])[0]
    pred_int  = np.argmax(probs)
    label     = encoder.inverse_transform([pred_int])[0]
    confidence = probs[pred_int] * 100

    # top 3 predictions
    top3_idx = np.argsort(probs)[::-1][:3]
    top3 = [(encoder.inverse_transform([i])[0], probs[i]*100) for i in top3_idx]

    print(f'Predicted: {label} ({confidence:.1f}% confidence)')
    print(f'Top 3: {top3[0][0]} ({top3[0][1]:.1f}%) | {top3[1][0]} ({top3[1][1]:.1f}%) | {top3[2][0]} ({top3[2][1]:.1f}%)')
    return label

print('--- fist (should be A/S/E) ---')
predict_sign({
    'flex_thumb': 200, 'flex_index': 100, 'flex_middle': 100,
    'flex_ring': 100, 'flex_pinky': 90
})

print('\n--- all fingers extended (should be 5) ---')
predict_sign({
    'flex_thumb': 550, 'flex_index': 550, 'flex_middle': 550,
    'flex_ring': 550, 'flex_pinky': 560
})

print('\n--- index finger up (should be 1/Z family) ---')
predict_sign({
    'flex_thumb': 145, 'flex_index': 550, 'flex_middle': 130,
    'flex_ring': 125, 'flex_pinky': 95
})

--- fist (should be A/S/E) ---
Predicted: A (96.9% confidence)
Top 3: A (96.9%) | S (2.5%) | N (0.5%)

--- all fingers extended (should be 5) ---
Predicted: 5 (100.0% confidence)
Top 3: 5 (100.0%) | C (0.0%) | 6 (0.0%)

--- index finger up (should be 1/Z family) ---
Predicted: Z (75.5% confidence)
Top 3: Z (75.5%) | 1 (24.5%) | X (0.0%)


c:\Users\gael8\cpe_design_1\helping-hand\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


np.str_('Z')